<a href="https://colab.research.google.com/github/FattahulAlim/Machine_Learning_Sem5/blob/main/JS06/Praktikum3_244107020018_MFattahulAlim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install hnswlib

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679799 sha256=74268f6b0a829c030bf171f2026e242e2b9ec3cd83fb590a70671bbfeba649a5
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50e3ef0b67e
Successfully built hnswlib


In [2]:
import hnswlib
import numpy as np
import time
from sklearn.neighbors import NearestNeighbors

# ===========================
# 1. Buat data 2D acak
# ===========================
num_elements = 1000
dim = 2
data = np.random.random((num_elements, dim)).astype(np.float32)

# Query point
query = np.array([[0.5, 0.5]], dtype=np.float32)
k = 5  # cari 5 tetangga terdekat

# ===========================
# 2. Exact NN (Brute Force)
# ===========================
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(data)

start = time.time()
distances, indices = nn.kneighbors(query)
end = time.time()

print("=== Exact NN ===")
print("Indices:", indices)
print("Distances:", distances)
print("Waktu:", end - start, "detik")

# ===========================
# 3. HNSW
# ===========================
# Inisialisasi index HNSW
p = hnswlib.Index(space='l2', dim=dim)

# Ukuran maksimum elemen yang bisa ditampung
p.init_index(max_elements=num_elements, ef_construction=100, M=16)

# Tambahkan data
p.add_items(data)

# Set parameter pencarian
p.set_ef(50)   # tradeoff speed vs accuracy

start = time.time()
labels, distances = p.knn_query(query, k=k)
end = time.time()

print("\n=== HNSW ===")
print("Indices:", labels)
print("Distances:", distances)
print("Waktu:", end - start, "detik")


=== Exact NN ===
Indices: [[217 628 660 342 926]]
Distances: [[0.03179516 0.03275166 0.04199332 0.04701397 0.04993226]]
Waktu: 0.029900550842285156 detik

=== HNSW ===
Indices: [[217 628 660 342 926]]
Distances: [[0.00101093 0.00107267 0.00176344 0.00221031 0.00249323]]
Waktu: 0.0001609325408935547 detik


In [3]:
def percobaan(metric, n, d, k=5, ef=50, M=16, ef_construction=100):
    n_repeat = 100 if n <= 1000 else 10
    rng = np.random.default_rng(42)
    data = rng.random((n, d), dtype=np.float32)
    query = np.full((1, d), 0.5, dtype=np.float32)

    # Samakan metric antara sklearn dan hnswlib
    if metric == 'l2':
        sk_metric = 'sqeuclidean'   # hnswlib 'l2' mengembalikan jarak kuadrat
        hn_space = 'l2'
    else:  # 'cosine'
        sk_metric = 'cosine'        # 1 - cosine similarity
        hn_space = 'cosine'

    # ---- Exact NN (brute force) ----
    nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric=sk_metric)
    nn.fit(data)
    t0 = time.perf_counter()
    for _ in range(n_repeat):
        dist_ex, idx_ex = nn.kneighbors(query)
    t_exact = (time.perf_counter() - t0) / n_repeat * 1000

    # ---- HNSW ----
    p = hnswlib.Index(space=hn_space, dim=d)
    p.init_index(max_elements=n, ef_construction=ef_construction, M=M)
    t0 = time.perf_counter()
    p.add_items(data)
    t_build = time.perf_counter() - t0
    p.set_ef(ef)

    t0 = time.perf_counter()
    for _ in range(n_repeat):
        labels, dist_hn = p.knn_query(query, k=k)
    t_hnsw = (time.perf_counter() - t0) / n_repeat * 1000

    recall = len(set(idx_ex[0]) & set(labels[0])) / k

    print(f"{metric} | n={n} | {d}D")
    print("  Index  :", idx_ex[0].tolist(), ",", labels[0].tolist())
    print(f"  Jarak  : {np.round(dist_ex[0], 4).tolist()} , {np.round(dist_hn[0], 4).tolist()}")
    print(f"  Waktu  : {t_exact:.4f} , {t_hnsw:.4f} ms")
    print(f"  Recall : {recall:.2f}")
    print(f"  (build HNSW: {t_build:.2f} detik)\n")

for metric in ['l2', 'cosine']:
    for n in [1000, 1_000_000]:
        for d in [2, 5]:
            percobaan(metric, n, d)

l2 | n=1000 | 2D
  Index  : [202, 358, 899, 883, 486] , [202, 358, 899, 883, 486]
  Jarak  : [0.0, 0.0004, 0.0009, 0.0011, 0.0012] , [0.0, 0.00039999998989515007, 0.0008999999845400453, 0.0010999999940395355, 0.0012000000569969416]
  Waktu  : 0.4984 , 0.0088 ms
  Recall : 1.00
  (build HNSW: 0.03 detik)

l2 | n=1000 | 5D
  Index  : [870, 383, 53, 512, 351] , [870, 383, 53, 512, 351]
  Jarak  : [0.0265, 0.0511, 0.0537, 0.0657, 0.0793] , [0.026499999687075615, 0.051100000739097595, 0.053700000047683716, 0.065700002014637, 0.07930000126361847]
  Waktu  : 0.3766 , 0.0230 ms
  Recall : 1.00
  (build HNSW: 0.04 detik)

l2 | n=1000000 | 2D
  Index  : [16541, 975941, 971854, 202, 820255] , [16541, 975941, 971854, 202, 820255]
  Jarak  : [0.0, 0.0, 0.0, 0.0, 0.0] , [0.0, 0.0, 0.0, 0.0, 0.0]
  Waktu  : 60.7664 , 0.0170 ms
  Recall : 1.00
  (build HNSW: 63.05 detik)

l2 | n=1000000 | 5D
  Index  : [416143, 972743, 511758, 357607, 291390] , [416143, 972743, 511758, 357607, 291390]
  Jarak  : [0.00